# Neural Scores and Hybrid Decision Pipeline Lab


## Purpose

This lab combines neural scores with symbolic rules.

Learning goals:

- Simulate neural risk scores.
- Apply symbolic constraints.
- Produce a hybrid decision.
- Track the decision source.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 500

records = pd.DataFrame({
    "entity_id": [f"E-{i:04d}" for i in range(1, n + 1)],
    "neural_risk_score": rng.beta(2.2, 4.0, size=n),
    "condition_score": rng.uniform(0.10, 0.98, size=n),
    "criticality": rng.choice(["low", "medium", "high"], size=n, p=[0.45, 0.35, 0.20]),
    "sensitive_workflow": rng.choice([0, 1], size=n, p=[0.80, 0.20]),
})

records["neural_recommendation"] = (records["neural_risk_score"] >= 0.55).astype(int)

records["symbolic_review_required"] = (
    ((records["criticality"] == "high") & (records["condition_score"] <= 0.35))
    | ((records["sensitive_workflow"] == 1) & (records["neural_risk_score"] >= 0.45))
    | ((records["criticality"] == "medium") & (records["condition_score"] <= 0.25))
).astype(int)

records["hybrid_decision"] = (
    (records["neural_recommendation"] == 1)
    | (records["symbolic_review_required"] == 1)
).astype(int)

records["decision_source"] = np.select(
    [
        (records["neural_recommendation"] == 1) & (records["symbolic_review_required"] == 1),
        (records["neural_recommendation"] == 1) & (records["symbolic_review_required"] == 0),
        (records["neural_recommendation"] == 0) & (records["symbolic_review_required"] == 1),
    ],
    [
        "neural_and_symbolic",
        "neural_only",
        "symbolic_only",
    ],
    default="no_review",
)

records.head()

In [ ]:
summary = (
    records
    .groupby("decision_source")
    .agg(
        n=("entity_id", "count"),
        mean_score=("neural_risk_score", "mean"),
        mean_condition=("condition_score", "mean"),
        review_rate=("hybrid_decision", "mean"),
    )
    .reset_index()
)

summary

## Interpretation

Hybrid diagnostics should show how often decisions come from neural scores, symbolic rules, or both.